Preprocessing: Removing headers and footers, fixing broken bullets, and normalizing text.1
Chunking: Splitting the text into paragraphs or chunks and assigning chunk IDs to produce document chunks with IDs.

In [ ]:
%pip install pypdf
%pip uninstall -y docx
%pip install python-docx

In [1]:
import re
import unicodedata
from collections import Counter
from pathlib import Path

import pandas as pd
from pypdf import PdfReader


MAX_CHUNK_CHARS = 1500

def read_document(path):
    """Extract text from PDF, DOCX, or TXT files."""
    if path.suffix.lower() == ".pdf":
        reader = PdfReader(str(path))
        return [page.extract_text() or "" for page in reader.pages]

    # if path.suffix.lower() == ".docx":
    #     document = Document(str(path))
    #     return ["\n".join(p.text for p in document.paragraphs)]

    # if path.suffix.lower() == ".txt":
    #     return [path.read_text(encoding="utf-8", errors="ignore")]

    return []


def normalize_text(text):
    """Normalize whitespace, bullets, and Unicode characters."""
    text = unicodedata.normalize("NFKC", text)
    text = text.replace("\u00ad", "")  # soft hyphen
    text = text.replace("\r\n", "\n").replace("\r", "\n")

    # Normalize common bullet characters
    text = re.sub(r"^[\s]*[•●▪◦‣⁃]\s*", "- ", text, flags=re.MULTILINE)

    # Repair bullets split across lines
    text = re.sub(r"(^-\s*)\n\s*", r"\1", text, flags=re.MULTILINE)

    # Remove excessive spaces while preserving paragraphs
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)

    return text.strip()


def remove_repeated_headers_footers(pages):
    """Remove lines repeated at the top or bottom of PDF pages."""
    page_lines = []

    for page in pages:
        lines = [line.strip() for line in page.splitlines() if line.strip()]
        page_lines.append(lines)

    candidates = []
    for lines in page_lines:
        candidates.extend(lines[:3])
        candidates.extend(lines[-3:])

    repeated_lines = {
        line for line, count in Counter(candidates).items()
        if count >= 2 and len(line) > 2
    }

    cleaned_pages = []
    for lines in page_lines:
        lines = [
            line for line in lines
            if line not in repeated_lines and not re.fullmatch(r"\d+", line)
        ]
        cleaned_pages.append("\n".join(lines))

    return cleaned_pages


def create_chunks(text, max_chars=MAX_CHUNK_CHARS):
    """Split normalized text into paragraph-based chunks."""
    paragraphs = [
        p.strip()
        for p in re.split(r"\n\s*\n", text)
        if p.strip()
    ]

    chunks = []
    current = ""

    for paragraph in paragraphs:
        if len(current) + len(paragraph) + 2 <= max_chars:
            current = f"{current}\n\n{paragraph}".strip()
        else:
            if current:
                chunks.append(current)

            # Split very long paragraphs by sentence
            if len(paragraph) > max_chars:
                sentences = re.split(r"(?<=[.!?])\s+", paragraph)
                current = ""

                for sentence in sentences:
                    if len(current) + len(sentence) + 1 <= max_chars:
                        current = f"{current} {sentence}".strip()
                    else:
                        if current:
                            chunks.append(current)
                        current = sentence
            else:
                current = paragraph

    if current:
        chunks.append(current)

    return chunks


def preprocess_document(input_file, output_file, max_chunk_chars=MAX_CHUNK_CHARS):
    """Extract, normalize, chunk, and save one PDF document."""
    input_file = Path(input_file)
    output_file = Path(output_file)
    output_file.parent.mkdir(parents=True, exist_ok=True)
    records = []

    file_path = input_file
    if file_path.suffix.lower() != ".pdf":
        raise ValueError(f"Expected a PDF input file, got: {file_path}")

    pages = read_document(file_path)

    pages = remove_repeated_headers_footers(pages)

    text = normalize_text("\n\n".join(pages))
    chunks = create_chunks(text, max_chars=max_chunk_chars)
    document_id = file_path.stem.lower().replace(" ", "_")

    for index, chunk in enumerate(chunks, start=1):
        records.append({
            "document_id": document_id,
            "chunk_id": f"{document_id}_{index:04d}",
            "source_file": file_path.name,
            "chunk_text": chunk,
        })

    chunks_df = pd.DataFrame(records)
    chunks_df.to_csv(output_file, index=False)
    return chunks_df




In [2]:
INPUT_FILE = Path("data/documents/Visa_Credit_Card_Agreement.pdf")
OUTPUT_FILE = "output/visa_credit_card_agreement_chunks.csv"
MAX_CHUNK_CHARS = 1500

chunks_df = preprocess_document(INPUT_FILE, OUTPUT_FILE, MAX_CHUNK_CHARS)
print(f"Created {len(chunks_df)} chunks.")
display(chunks_df.head())

Created 25 chunks.


,document_id,chunk_id,source_file,chunk_text
0,visa_credit_card_agreement,visa_credit_card_agreement_0001,Visa_Credit_Card_Agreement.pdf,"©CUNA Mutual Group 1991, 2006, 09, 10, 12 All ..."
1,visa_credit_card_agreement,visa_credit_card_agreement_0002,Visa_Credit_Card_Agreement.pdf,The Credit Union may increase or decrea se you...
2,visa_credit_card_agreement,visa_credit_card_agreement_0003,Visa_Credit_Card_Agreement.pdf,Written stop pay ment orders are effective onl...
3,visa_credit_card_agreement,visa_credit_card_agreement_0004,Visa_Credit_Card_Agreement.pdf,If you wish to\npay for goods or services over...
4,visa_credit_card_agreement,visa_credit_card_agreement_0005,Visa_Credit_Card_Agreement.pdf,PROMISE TO PAY — You promise to pay all charge...


In [28]:
display(chunks_df.chunk_text.values[:10])

array(['©CUNA Mutual Group 1991, 2006, 09, 10, 12 All Rights Reserved DGAEL6 (MXC501 CCM002)-e\nVISA\nCONSUMER CREDIT CARD AGREEMENT\nIn this Agreement, “Agreement” means this Consumer Cred it Card Agreement. “Disclosure” means the Credit Card\nAccount Opening Disclosure. The Account Opening Disclosure is incorporated into this Consumer Credit Card Agreement\nand is part of the Agreement. In this Agreement the word s "you," “your,” and "yours" me an each and all of those who\nagree to be bound by this Agreement; "card" means the Visa credit card and any duplicates, renewals, or substitutions the\nCredit Union issues to you; "account" means your Visa credit card line-of-credit account with the Credit Union; and "Credit\nUnion" means the Credit Union whose name appears on this Agreement or anyone to whom the Credit Union transfers\nthis Agreement. 1. USING YOUR ACCOUNT — If you are approved for an account, the Credit Union will establish a line of credit for you. You agree that your cred

In [22]:
INPUT_FILE = Path("data/documents/cardholder-agreement-en.pdf")
OUTPUT_FILE = "output/cardholder_agreement_chunks.csv"
MAX_CHUNK_CHARS = 1500

chunks_df = preprocess_document(INPUT_FILE, OUTPUT_FILE, MAX_CHUNK_CHARS)
print(f"Created {len(chunks_df)} chunks.")
display(chunks_df.head())

Created 41 chunks.


,document_id,chunk_id,source_file,chunk_text
0,cardholder-agreement-en,cardholder-agreement-en_0001,cardholder-agreement-en.pdf,"CIBC CARDHOLDER AGREEMENT\nEffective August 1,..."
1,cardholder-agreement-en,cardholder-agreement-en_0002,cardholder-agreement-en.pdf,1. Introduction\na) This Agreement applies to ...
2,cardholder-agreement-en,cardholder-agreement-en_0003,cardholder-agreement-en.pdf,If there is a conflict between this Agreement ...
3,cardholder-agreement-en,cardholder-agreement-en_0004,cardholder-agreement-en.pdf,"At a minimum, this means you will:\n- sign and..."
4,cardholder-agreement-en,cardholder-agreement-en_0005,cardholder-agreement-en.pdf,d) Manage Y our Pre-Authorized Payments. You c...


In [4]:
%pip install tiktoken

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 62.2 MB/s  0:00:00
Note: you may need to restart the kernel to use updated packages.


In [5]:

from pathlib import Path

import fitz
import tiktoken


# Configuration
INPUT_FILE = Path(
    "data/documents/Visa_Credit_Card_Agreement.pdf",
)
OUTPUT_FILE = Path(
    "output/visa_credit_card_agreement_chunks.csv",
)

MAX_CHUNK_TOKENS = 400
CHUNK_OVERLAP = 60
MIN_CHUNK_TOKENS = 80

encoding = tiktoken.get_encoding("cl100k_base")


def count_tokens(text):
    return len(encoding.encode(text))


# Step 1: Extract text with page metadata
def extract_pages(pdf_path):
    pages = []

    with fitz.open(pdf_path) as doc:
        for page_num, page in enumerate(doc, start=1):
            text = page.get_text("text")

            # Normalize whitespace without
            # destroying paragraph boundaries
            text = text.replace("\r\n", "\n")
            text = re.sub(r"[ \t]+", " ", text)
            text = re.sub(r"\n{3,}", "\n\n", text)

            pages.append({
                "page": page_num,
                "text": text.strip(),
            })

    return pages


# Step 2: Identify document sections
def extract_sections(pages):
    sections = []
    current = None

    # Match numbered legal headings
    heading_pattern = re.compile(
        r"(?m)^\s*(\d{1,2})\.\s+"
        r"([A-Z][A-Z0-9 /(),\-–]+?)\s*[—–-]\s*",
    )

    for page in pages:
        text = page["text"]
        matches = list(heading_pattern.finditer(text))

        position = 0

        for match in matches:
            # Text before next heading belongs
            # to the previous section
            preceding = text[position:match.start()].strip()

            if preceding:
                if current is None:
                    current = {
                        "section": "Introduction",
                        "page_start": page["page"],
                        "page_end": page["page"],
                        "text": "",
                    }

                current["text"] += "\n" + preceding
                current["page_end"] = page["page"]

            if current is not None:
                sections.append(current)

            current = {
                "section": (
                    f"{match.group(1)}. "
                    f"{match.group(2).strip()}"
                ),
                "page_start": page["page"],
                "page_end": page["page"],
                "text": "",
            }

            position = match.end()

        remaining = text[position:].strip()

        if remaining:
            if current is None:
                current = {
                    "section": "Introduction",
                    "page_start": page["page"],
                    "page_end": page["page"],
                    "text": "",
                }

            current["text"] += "\n" + remaining
            current["page_end"] = page["page"]

    if current is not None:
        sections.append(current)

    return sections


# Step 3: Token-based recursive chunking
def split_into_chunks(
    text,
    max_tokens=400,
    overlap=60,
):
    if not text.strip():
        return []

    if count_tokens(text) <= max_tokens:
        return [text.strip()]

    # Prefer paragraph boundaries
    paragraphs = re.split(r"\n\s*\n", text)

    # Fall back to sentence splitting
    units = []

    for paragraph in paragraphs:
        paragraph = paragraph.strip()

        if not paragraph:
            continue

        if count_tokens(paragraph) <= max_tokens:
            units.append(paragraph)
        else:
            sentences = re.split(
                r"(?<=[.!?])\s+",
                paragraph,
            )
            units.extend(sentences)

    # Ensure even very long sentences fit
    safe_units = []

    for unit in units:
        tokens = encoding.encode(unit)

        if len(tokens) <= max_tokens:
            safe_units.append(unit)
        else:
            for i in range(0, len(tokens), max_tokens):
                safe_units.append(
                    encoding.decode(
                        tokens[i:i + max_tokens],
                    ),
                )

    chunks = []
    current_units = []
    current_tokens = 0

    for unit in safe_units:
        unit_tokens = count_tokens(unit)

        if (
            current_units
            and current_tokens + unit_tokens > max_tokens
        ):
            chunks.append(
                "\n\n".join(current_units).strip(),
            )

            # Preserve trailing context
            previous = current_units[:]
            current_units = []
            current_tokens = 0

            for item in reversed(previous):
                n = count_tokens(item)

                if current_tokens + n > overlap:
                    break

                current_units.insert(0, item)
                current_tokens += n

            # Avoid an oversized chunk
            if current_tokens + unit_tokens > max_tokens:
                current_units = []
                current_tokens = 0

        current_units.append(unit)
        current_tokens += unit_tokens

    if current_units:
        chunks.append(
            "\n\n".join(current_units).strip(),
        )

    return chunks


# Step 4: Build dataset
def preprocess_document(input_file, output_file):
    pages = extract_pages(input_file)
    sections = extract_sections(pages)

    records = []
    chunk_id = 0

    for section in sections:
        chunks = split_into_chunks(
            section["text"],
            max_tokens=MAX_CHUNK_TOKENS,
            overlap=CHUNK_OVERLAP,
        )

        for index, chunk in enumerate(chunks):
            records.append({
                "chunk_id": chunk_id,
                "source": input_file.name,
                "section": section["section"],
                "page_start": section["page_start"],
                "page_end": section["page_end"],
                "chunk_index": index,
                "token_count": count_tokens(chunk),
                "text": chunk,
            })

            chunk_id += 1

    df = pd.DataFrame(records)

    output_file.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    df.to_csv(output_file, index=False)

    return df


# Step 5: Run
chunks_df = preprocess_document(
    INPUT_FILE,
    OUTPUT_FILE,
)

print(f"Created {len(chunks_df)} chunks")
display(chunks_df.head())


Created 36 chunks


,chunk_id,source,section,page_start,page_end,chunk_index,token_count,text
0,0,Visa_Credit_Card_Agreement.pdf,Introduction,1,1,0,201,"©CUNA Mutual Group 1991, 2006, 09, 10, 12 All ..."
1,1,Visa_Credit_Card_Agreement.pdf,1. USING YOUR ACCOUNT,1,1,0,386,"If you are approved for an account, the Credit..."
2,2,Visa_Credit_Card_Agreement.pdf,1. USING YOUR ACCOUNT,1,1,1,220,"If permitted, you may make a stop payment requ..."
3,3,Visa_Credit_Card_Agreement.pdf,2. USING YOUR CARD,1,1,0,354,You understand that the use of your credit car...
4,4,Visa_Credit_Card_Agreement.pdf,3. PROMISE TO PAY,1,2,0,110,"You promise to pay all charges (purchases, cas..."
